### Tarea #2 Gema Guerra Valdez (1819110)

#### Requisitos

- Elegir un conjunto de datos de reseñas de usuarios (por ejemplo, Reseñas de Amazon, entre otros).  

- Elegir un método de vectorización adecuado para el conjunto de datos elegido, aplicarlo y estudiar las propiedades de los vectores obtenidos.  

- Realizar el preprocesamiento necesario para realizar un análisis de sentimiento que, de ser posible, compare la reseña presentada con alguna calificación numérica elegida.  

- Escribir un reporte con los hallazgos, metodología y resultados en PDF y subirlo en una sección claramente identificable de tu repositorio.

In [ ]:
pip install transformers datasets torch scikit-learn pandas accelerate

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [ ]:
dataset = load_dataset("Yelp/yelp_review_full")

In [ ]:
print(dataset)

In [ ]:
train_set = dataset["train"].shuffle(seed=42).select(range(2000))
test_set = dataset["test"].shuffle(seed=42).select(range(500))

In [ ]:
# Bolsa de Palabras
print("Analizando Enfoque de Bolsa de Palabras")
corpus_ejemplo = train_set["text"][:500]  # Submuestra para el análisis vectorial

vectorizer = CountVectorizer(stop_words='english', max_features=1000)
bow_vectors = vectorizer.fit_transform(corpus_ejemplo)

# Propiedades
print(f"Forma de la matriz BoW (Documentos, Vocabulario): {bow_vectors.shape}")
elementos_no_cero = bow_vectors.nnz
total_elementos = bow_vectors.shape[0] * bow_vectors.shape[1]
escasez_sparsity = (1.0 - (elementos_no_cero / total_elementos)) * 100
print(f"Propiedad de Escasez (Sparsity): {escasez_sparsity:.2f}% de los elementos son ceros.")

In [ ]:
# Preprocesamiento y Tokenización con BERT
# 'bert-base-uncased' (ideal para inglés en minúsculas)
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # longitud máxima fija 128 tokens para ahorrar memoria
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizando los textos...")
tokenized_train = train_set.map(tokenize_function, batched=True)
tokenized_test = test_set.map(tokenize_function, batched=True)

In [ ]:
# Modelo
# Yelp tiene 5 clases de calificaciones (0, 1, 2, 3, 4)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

# Definimos los argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./results",          # Carpeta de salida
    learning_rate=2e-5,              # Tasa de aprendizaje típica para BERT
    per_device_train_batch_size=8,   # Batch size (ajustar según tu GPU/VRAM)
    per_device_eval_batch_size=8,
    num_train_epochs=2,              # 2 o 3 épocas suelen ser suficientes
    weight_decay=0.01,
    eval_strategy="epoch",     # Evaluar al final de cada época
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=50,
    disable_tqdm=True
)

# Función auxiliar para calcular métricas durante el entrenamiento
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": (preds == labels).astype(np.float32).mean().item()}

# Inicializamos el Trainer de Hugging Face
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Iniciando el entrenamiento (Fine-Tuning)...")
trainer.train()

In [ ]:
# Comparativa
# Realizar predicciones
predictions = trainer.predict(tokenized_test)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = tokenized_test["label"]

# Generar reporte de clasificación
print("Clasificación Final (Comparación Predicción vs Calificación Real):")
report = classification_report(y_true, y_pred, target_names=["1 Estrella", "2 Estrellas", "3 Estrellas", "4 Estrellas", "5 Estrellas"])
print(report)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
labels = ["1 Estrella", "2 Estrellas", "3 Estrellas", "4 Estrellas", "5 Estrellas"]
plt.figure(figsize=(8, 6))
sns.set_theme(style="white")

# mapa de calor
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels,
            cbar_kws={'label': 'Número de Reseñas'})

plt.title("Matriz de Confusión: Fine-Tuning de BERT en Yelp", fontsize=14, pad=15)
plt.xlabel("Clase Predicha por el Modelo", fontsize=12, labelpad=10)
plt.ylabel("Clase Real (Calificación Usuario)", fontsize=12, labelpad=10)
plt.tight_layout()

# 4. Guardar la imagen para tu reporte en PDF
# plt.savefig("matriz_confusion.png", dpi=300)
plt.show()